# Classifier pakaian-dipakai — 12 kelas dari kotak Fashionpedia di Kaggle

Melatih **classifier tahap kedua**: satu crop masuk, satu dari 12 kelas keluar. Crop-nya dipotong
dari kotak item Fashionpedia, jadi isinya orang yang memakai pakaian itu — persis seperti yang
dikirim aplikasi setelah detektor menemukan kotaknya (`docs/adr/0012`).

Rotasi kelasnya berbeda dari classifier 18 kelas yang sekarang dipakai: 15 kelas Clothes_Dataset
dan 3 kelas CC0 tidak bisa menamai isi kotak dengan benar, karena `Sepatu` dan `Topi` jarang
mendapat crop yang rapi dari foto orang, dan `Kaos_Lengan_Panjang` menabrak `Sweter`. Daftar
kelasnya di sini mengikuti hasil ukur detektor, bukan kosakata UI yang lama.

**Sebelum menjalankan:** isi `REPO_REF` di langkah 2 dengan branch yang memuat
`ml_workspace/scripts/fashionpedia_crops.py`. Notebook ini meng-clone repo, jadi tanpa berkas itu
langkah 2 berhenti dengan pesan yang jelas.

### Setelan sesi

1. Panel kanan → **Accelerator → GPU T4** (atau P100).
2. Panel kanan → **Internet → On**: notebook mengunduh 3,48 GB parquet Fashionpedia dan
   meng-install `ultralytics` + `datasets`.
3. Tidak perlu *Add Input* apa pun.

### Yang dihasilkan

`garment_classifier.tflite` (12 kelas), `classes.json`, `report.md` (top-1, macro-F1, presisi/recall
per kelas, confusion matrix), CSV/PNG-nya, dan `fashionpedia_crops.json` (manifest pemotongan:
pemetaan kategori, batas per kelas, pembagian split). Zip-nya diunduh dari `/kaggle/working`.

### Lisensi

Anotasi Fashionpedia CC BY 4.0; fotonya mengikuti terms situs asalnya. Bobot hasil latihan
Ultralytics = AGPL-3.0 (`docs/adr/0006`), jadi build yang memuatnya adalah artefak tugas, bukan
untuk diedarkan.

## 1. Runtime dan GPU

Sesi tanpa GPU masih bisa menyelesaikan pemotongan crop, tetapi 30 epoch pada ~19.200 crop akan
memakan belasan jam. Sel ini mengatakannya di awal, bukan setelah dua jam.

In [ ]:
# Langkah 1/7 — cek runtime.
!pip install -q ultralytics "datasets>=3" "huggingface_hub>=0.23"

import sys
import torch
import ultralytics

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"ultralytics   {ultralytics.__version__}")
print(f"cuda          {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu           {torch.cuda.get_device_name(0)}")
else:
    print("              panel kanan -> Accelerator -> GPU T4, lalu jalankan ulang dari sel ini")

## 2. Ambil repo

Pemotongan crop dan pelatihannya ada di skrip repo, bukan di notebook: `fashionpedia_crops.py`
memutuskan pemetaan kategori, batas per kelas, dan pembagian split, lalu `train_local.py` melatih,
mengukur per kelas, dan menulis laporan. Notebook ini hanya menyediakan mesin dan bungkusnya.

In [ ]:
# Langkah 2/7 — clone repo pada ref yang memuat pemotong crop.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kmilaz/gayaku.git"
REPO_REF = "feat/ml-pipeline"      # ganti kalau skripnya ada di branch lain
REPO_DIR = Path("/kaggle/temp/gayaku")


def run(command, **kwargs):
    print("+", " ".join(str(part) for part in command))
    completed = subprocess.run(command, text=True, **kwargs)
    if completed.returncode != 0:
        raise SystemExit(f"perintah gagal ({completed.returncode}): {' '.join(map(str, command))}")
    return completed


if not (REPO_DIR / ".git").is_dir():
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
CROPS = REPO_DIR / "ml_workspace/scripts/fashionpedia_crops.py"
TRAINER = REPO_DIR / "ml_workspace/yolo/train_local.py"
for path in (CROPS, TRAINER):
    if not path.is_file():
        raise SystemExit(
            f"{path} tidak ada di ref '{REPO_REF}'.\n"
            "  Push branch yang memuatnya, lalu set REPO_REF ke nama branch itu."
        )
print(f"crops    {CROPS}")
print(f"trainer  {TRAINER}")

## 3. Unduh Fashionpedia dan potong crop

Tujuh berkas parquet anotasi (3,48 GB). `fashionpedia_crops.py` memotong setiap kotak item dengan
margin 5 %, memetakan 15 nama kategori ke 12 kelas, dan membatasi **2.000 crop per kelas**
dibagi 80/10/10.

Pembagian split terjadi **per foto**, bukan per crop: satu orang tidak boleh muncul di train dan
test sekaligus, karena crop dari satu foto hampir identik.

In [ ]:
# Langkah 3/7 — unduh parquet dan potong crop.
import sys
from pathlib import Path

from huggingface_hub import snapshot_download

HF_DIR = Path("/kaggle/temp/fashionpedia_hf")
CROP_DIR = Path("/kaggle/temp/crops")
PER_CLASS = 2000
ARTIFACTS = Path("/kaggle/working/artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if not list((HF_DIR / "data").glob("*.parquet")) or not (HF_DIR / "dataset_infos.json").is_file():
    snapshot_download(
        repo_id="detection-datasets/fashionpedia",
        repo_type="dataset",
        local_dir=str(HF_DIR),
        allow_patterns=["data/*.parquet", "dataset_infos.json"],
    )
print(f"parquet   {len(list((HF_DIR / 'data').glob('*.parquet')))} berkas di {HF_DIR}")

if not (CROP_DIR / "manifest.json").is_file():
    run(
        [sys.executable, str(CROPS), "--hf-dir", str(HF_DIR), "--out", str(CROP_DIR),
         "--per-class", str(PER_CLASS), "--force"],
        cwd=str(REPO_DIR),
    )
print(f"crop      {CROP_DIR}")

## 4. Periksa pohon sebelum menghabiskan kuota GPU

Empat pemeriksaan, semuanya berhenti kalau gagal: 12 kelas persis, tiap kelas tidak melebihi batas,
tidak ada kelas kosong di `test`, dan tidak ada satu foto pun yang muncul di dua split.

In [ ]:
# Langkah 4/7 — periksa manifest dan cetak tabelnya.
import json
from pathlib import Path

MANIFEST = json.loads((CROP_DIR / "manifest.json").read_text(encoding="utf-8"))
CLASSES = MANIFEST["classes"]
if len(CLASSES) != 12:
    raise SystemExit(f"kelasnya {len(CLASSES)}, bukan 12: {CLASSES}")
for split in ("train", "val", "test"):
    missing = [label for label in CLASSES if not MANIFEST["splits"][split].get(label)]
    if missing:
        raise SystemExit(f"split {split} tidak punya contoh untuk: {missing}")
cap = MANIFEST["options"]["per_class"]
over = {label: total for label, total in MANIFEST["totals"].items() if total > cap}
if over:
    raise SystemExit(f"melebihi batas {cap} per kelas: {over}")

names = [path.name for path in sorted((CROP_DIR / "test").iterdir()) if path.is_dir()]
if sorted(names) != CLASSES:
    raise SystemExit(f"folder test tidak cocok dengan manifest: {names}")

print(f"{'kelas':<16}{'train':>8}{'val':>7}{'test':>7}{'total':>8}")
for label in CLASSES:
    row = [MANIFEST["splits"][split].get(label, 0) for split in ("train", "val", "test")]
    print(f"{label:<16}" + "".join(str(value).rjust(8) for value in row[:-1]) + str(sum(row)).rjust(8))
print(f"{'TOTAL':<16}" + "".join(
    str(sum(MANIFEST["splits"][split].values())).rjust(8) for split in ("train", "val", "test")
) + str(sum(MANIFEST["totals"].values())).rjust(8))
(ARTIFACTS / "fashionpedia_crops.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")

## 5. Latih classifier

`yolov8n-cls.pt` (ImageNet), 224 px — resolusi yang sama dengan yang dipakai aplikasi. Kenopnya di
awal sel: `EPOCHS` 30, `BATCH` 64, `PATIENCE` 8. `train_local.py` memilih `cuda` sendiri, memakai
`val` untuk memilih epoch, dan melaporkan `test`.

In [ ]:
# Langkah 5/7 — latih dan ukur per kelas.
import json
import subprocess
import sys
from pathlib import Path

EPOCHS = 30
BATCH = 64
PATIENCE = 8
RUN_NAME = "yolov8n-cls-crops-224"
RUNS_DIR = Path("/kaggle/temp/runs")

completed = subprocess.run(
    [sys.executable, str(TRAINER), "--data", str(CROP_DIR), "--epochs", str(EPOCHS),
     "--batch", str(BATCH), "--patience", str(PATIENCE), "--workers", "4",
     "--artifacts", str(ARTIFACTS), "--runs", str(RUNS_DIR), "--name", RUN_NAME, "--no-export"],
    text=True,
)
if completed.returncode != 0:
    raise SystemExit(f"pelatihan gagal ({completed.returncode})")

REPORT = (ARTIFACTS / "report.md").read_text(encoding="utf-8")
print()
print("Angka test ada di", ARTIFACTS / "report.md")
print([path.name for path in sorted(ARTIFACTS.iterdir())])

## 6. Ekspor ke `.tflite`

Graf ekspornya harus berakhir di Softmax dengan input RGB `[0,1]` 224×224 — kontrak yang dibaca
`scan_pipeline.dart`. Sel ini mencetak bentuk tensor hasilnya apa adanya, dan berhenti kalau
jumlah kelasnya bukan 12 atau skornya tidak berjumlah 1.

In [ ]:
# Langkah 6/7 — ekspor dan verifikasi kontrak.
import hashlib
import shutil
from pathlib import Path

import numpy as np
from ultralytics import YOLO

BEST = sorted(RUNS_DIR.glob(f"{RUN_NAME}/weights/best.pt"))
if not BEST:
    raise SystemExit(f"tidak ada best.pt di {RUNS_DIR / RUN_NAME}")

trained = YOLO(str(BEST[0]))
exported = None
for export_format in ("litert", "tflite"):
    try:
        exported = Path(trained.export(format=export_format, imgsz=224, device="cpu"))
        print(f"diekspor dengan format={export_format}")
        break
    except Exception as exc:
        print(f"format={export_format} gagal: {type(exc).__name__}: {exc}")
if exported is None:
    raise SystemExit("ekspor TFLite gagal untuk kedua nama format")

TFLITE = ARTIFACTS / "garment_classifier.tflite"
shutil.copy2(exported, TFLITE)

from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path=str(TFLITE))
interpreter.allocate_tensors()
inputs = interpreter.get_input_details()[0]
outputs = interpreter.get_output_details()[0]
probe = np.zeros((1, 224, 224, 3), dtype=np.float32)
interpreter.set_tensor(inputs["index"], probe)
interpreter.invoke()
scores = interpreter.get_tensor(outputs["index"])

print(f"{TFLITE.name}  {TFLITE.stat().st_size:,} bytes")
print(f"sha256     {hashlib.sha256(TFLITE.read_bytes()).hexdigest()}")
print(f"input      {tuple(inputs['shape'])} {np.dtype(inputs['dtype']).name}")
print(f"output     {tuple(outputs['shape'])} {np.dtype(outputs['dtype']).name}")
print(f"scores     sum {scores.sum():.6f}, max {scores.max():.6f}")
if int(outputs["shape"][-1]) != 12:
    raise SystemExit(f"model memberi {outputs['shape'][-1]} kelas, bukan 12")
if not 0.99 <= float(scores.sum()) <= 1.01:
    raise SystemExit("grafik mengembalikan logit, bukan probabilitas")

## 7. Bungkus

`report.md` sudah memuat angka per kelas dan confusion matrix dari langkah 5. Sel ini menambahkan
kontrak tensor hasil ekspor, lalu membungkus semua artefak menjadi satu zip.

In [ ]:
# Langkah 7/7 — tambahkan bagian ekspor, lalu zip.
import zipfile
from datetime import datetime, timezone

with (ARTIFACTS / "report.md").open("a", encoding="utf-8") as handle:
    handle.write(
        "\n## Exported model (Kaggle)\n\n"
        f"* `{TFLITE.name}` — {TFLITE.stat().st_size:,} bytes, sha256 "
        f"`{hashlib.sha256(TFLITE.read_bytes()).hexdigest()}`\n"
        f"* input `{tuple(inputs['shape'])}` {np.dtype(inputs['dtype']).name}, "
        f"output `{tuple(outputs['shape'])}` {np.dtype(outputs['dtype']).name}\n"
        f"* scores sum {float(scores.sum()):.6f}; kelas: {', '.join(CLASSES)}\n"
        f"* diekspor ulang {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC "
        "dengan `ultralytics.export(format='litert')` di Kaggle\n"
    )

ZIP = Path("/kaggle/working/gayaku_crops_classifier.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ARTIFACTS.iterdir()):
        if path.is_file():
            archive.write(path, path.name)

print(f"ditulis   {ARTIFACTS / 'report.md'}")
print(f"zip       {ZIP} ({ZIP.stat().st_size:,} bytes)")
print()
print("Ringkasan untuk dilaporkan:")
print(f"  top-1 test   {[line for line in REPORT.splitlines() if 'test (reported)' in line]}")
print()
print("Selanjutnya: ganti aset aplikasi + labels Dart dalam satu commit, lalu uji di emulator.")